In [1]:
import pandas as pd
import numpy as np

raw_data = {
    'Full_Name': ['Krish', 'Amit Shah', 'Priya Patel', 'Rahul V.', 'Sanya M.', 'CEO_Dave', 'Anjali R.', 'Deepak K.'],
    'Role': ['ML Engineer', 'ml engineer', 'Data Scientist', 'data scientist', 'ML Eng', 'Founder', 'Data Scientist', 'ML Engineer'],
    'Experience_Years': [1, 3, 5, np.nan, 2, 20, 4, 1],
    'Location': ['Ahmedabad', 'Ahmedabad', 'Bangalore', 'Bangalore', 'Ahmedabad', 'Mumbai', 'Mumbai', 'Ahmedabad'],
    'Salary_INR': [1200000, 1300000, 1800000, 1750000, 1100000, 95000000, 1600000, 1250000]
}

df = pd.DataFrame(raw_data)
# Remove the CEO/Founder to focus only on Engineer salaries
df = df[df['Salary_INR'] < 5000000]
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 7 entries, 0 to 7
Data columns (total 5 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Full_Name         7 non-null      object 
 1   Role              7 non-null      object 
 2   Experience_Years  6 non-null      float64
 3   Location          7 non-null      object 
 4   Salary_INR        7 non-null      int64  
dtypes: float64(1), int64(1), object(3)
memory usage: 336.0+ bytes


In [2]:
df.describe()

,Experience_Years,Salary_INR
count,6.000000,7.000000e+00
mean,2.666667,1.428571e+06
std,1.632993,2.826322e+05
min,1.000000,1.100000e+06
25%,1.250000,1.225000e+06
50%,2.500000,1.300000e+06
75%,3.750000,1.675000e+06
max,5.000000,1.800000e+06


In [3]:
df['Role'].value_counts()

Role
ML Engineer       2
Data Scientist    2
ml engineer       1
data scientist    1
ML Eng            1
Name: count, dtype: int64

In [4]:
df["Role"]=df["Role"].str.lower().str.replace("engineer","eng").str.strip()
print(df["Role"])

0            ml eng
1            ml eng
2    data scientist
3    data scientist
4            ml eng
6    data scientist
7            ml eng
Name: Role, dtype: object


In [5]:
import numpy as np
m=df["Experience_Years"].median()
df["Experience_Years"] = df["Experience_Years"].fillna(m)
df["Experience_Years"]

0    1.0
1    3.0
2    5.0
3    2.5
4    2.0
6    4.0
7    1.0
Name: Experience_Years, dtype: float64

In [6]:
from sklearn.preprocessing import RobustScaler
robust_s_value=RobustScaler()
# This keeps the index perfectly aligned
df["Salary_INR"] = robust_s_value.fit_transform(df[["Salary_INR"]])
df = df.reset_index(drop=True)
print(df)

     Full_Name            Role  Experience_Years   Location  Salary_INR
0        Krish          ml eng               1.0  Ahmedabad   -0.222222
1    Amit Shah          ml eng               3.0  Ahmedabad    0.000000
2  Priya Patel  data scientist               5.0  Bangalore    1.111111
3     Rahul V.  data scientist               2.5  Bangalore    1.000000
4     Sanya M.          ml eng               2.0  Ahmedabad   -0.444444
5    Anjali R.  data scientist               4.0     Mumbai    0.666667
6    Deepak K.          ml eng               1.0  Ahmedabad   -0.111111


In [7]:
df_encode=pd.get_dummies(df,drop_first=True,columns=["Role","Location"])
df_encode

,Full_Name,Experience_Years,Salary_INR,Role_ml eng,Location_Bangalore,Location_Mumbai
0,Krish,1.0,-0.222222,True,False,False
1,Amit Shah,3.0,0.000000,True,False,False
2,Priya Patel,5.0,1.111111,False,True,False
3,Rahul V.,2.5,1.000000,False,True,False
4,Sanya M.,2.0,-0.444444,True,False,False
5,Anjali R.,4.0,0.666667,False,False,True
6,Deepak K.,1.0,-0.111111,True,False,False


In [8]:
from sklearn.model_selection import train_test_split
X=df_encode.drop(['Salary_INR', 'Full_Name'], axis=1)
y=df_encode["Salary_INR"]
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42)
print(X_train.shape)
print(X_test.shape)

(5, 4)
(2, 4)


In [9]:
from sklearn.linear_model import LinearRegression
model=LinearRegression()
model.fit(X_train,y_train)
ypred=model.predict(X_test)
n_df=pd.DataFrame({"Actual ": y_test,"Predict ":ypred})
n_df

,Actual,Predict
0,-0.222222,-0.273946
1,0.000000,-0.289272


In [10]:
from sklearn.metrics import *
print(r2_score(y_test,ypred))

-2.4973246135552896


In [11]:
real_predictions = robust_s_value.inverse_transform(ypred.reshape(-1, 1))
print(real_predictions)

[[1176724.13793103]
 [1169827.5862069 ]]


In [14]:
# 1. Look at your X_train columns to see the EXACT names
print(X_train.columns)

# 2. Use those EXACT names in your prediction dataframe
my_data = pd.DataFrame({
    'Experience_Years': [1.0],
    'Role_ml eng': [1],        # Make sure this matches the print above!
    
    'Location_Bangalore': [0],
    'Location_Mumbai': [0]
})

# 3. Predict
my_prediction = model.predict(my_data)
print(f"Your Scaled Salary Prediction: {my_prediction[0]}")

# 4. Convert back to real Rupees
real_inr = robust_s_value.inverse_transform(my_prediction.reshape(-1, 1))
print(f"Predicted Salary in INR: {real_inr[0][0]}")

Index(['Experience_Years', 'Role_ml eng', 'Location_Bangalore',
       'Location_Mumbai'],
      dtype='object')
Your Scaled Salary Prediction: -0.27394636015325746
Predicted Salary in INR: 1176724.1379310342
